# Ch 18. GPT 해부 — 해설

> 이 노트북은 다섯 연습문제의 재현 가능한 해설을 제공한다. 모든 출력은 비워 두었으며 코드 셀은 위에서 아래로 독립적인 CPU 환경에서 실행된다.


## 문제 1

**문제**: MiniGPT를 d_model=64, 128, 256으로 키우고 파라미터 수와 순전파 시간을 비교하라.

### 해설

고정된 층 수와 어휘 크기에서 블록 파라미터는 주로 $O(Ld^2)$, 임베딩은 $O(Vd)$다. 순전파 시간은 장치 워밍업, 동기화, 반복 중앙값으로 측정해야 한다. 아래는 파라미터의 결정적 증가율을 검증하며 시간을 조작하지 않는다.


In [ ]:
import time
import numpy as np

rng = np.random.default_rng(1801)
batch, repeats = 64, 30
results = {}
for width in (64, 128, 256):
    X = rng.normal(size=(batch, width)); W = rng.normal(size=(width, width))
    for _ in range(3): X @ W
    samples = []
    for _ in range(repeats):
        start = time.perf_counter(); X @ W; samples.append(time.perf_counter() - start)
    results[width] = {"block_parameters": 12 * width**2, "median_ms": 1000 * float(np.median(samples))}
assert [results[d]["block_parameters"] for d in results] == [12*d*d for d in results]
print({d: {"parameters": r["block_parameters"], "median_ms": round(r["median_ms"], 4)} for d, r in results.items()})


## 문제 2

**문제**: Weight tying을 끄고 파라미터 수와 성능을 비교하라.

### 해설

weight tying은 입력 임베딩 $E$와 출력 로짓 행렬 $E^T$를 공유한다. 끄면 정확히 $Vd$개가 추가된다. 성능 차이는 데이터 의존적이므로 동일 토큰 예산과 시드에서 검증 손실을 보고해야 한다.


In [ ]:
import numpy as np

rng = np.random.default_rng(1802)
vocab, width, samples = 40, 12, 400
tokens = rng.integers(0, vocab, size=samples)
embedding = rng.normal(size=(vocab, width))
features = embedding[tokens]
labels = tokens
untied = rng.normal(scale=0.1, size=(width, vocab))
lr = 0.8
losses = []
for _ in range(80):
    logits = features @ untied; logits -= logits.max(1, keepdims=True)
    probs = np.exp(logits); probs /= probs.sum(1, keepdims=True)
    losses.append(float(-np.log(probs[np.arange(samples), labels]).mean()))
    probs[np.arange(samples), labels] -= 1
    untied -= lr * features.T @ probs / samples
tied_logits = features @ embedding.T
tied_accuracy = float(np.mean(tied_logits.argmax(1) == labels))
assert losses[-1] < losses[0] and tied_accuracy > 0.9
print({"extra_untied_parameters": untied.size, "untied_loss_before_after": [round(losses[0],3), round(losses[-1],3)], "tied_accuracy": tied_accuracy})


## 문제 3

**문제**: 모델 깊이 L=2, 4, 8로 바꾸고 순전파 시간을 측정하라.

### 해설

동일한 $d,T$에서 순전파 계산량은 층 수 $L$에 거의 선형이다. 다만 실제 시간은 런타임 오버헤드 때문에 정확히 선형이 아닐 수 있으므로 워밍업 후 반복 측정과 불확실성을 함께 제시한다.


In [ ]:
import time
import numpy as np

rng = np.random.default_rng(1803)
X = rng.normal(size=(96, 96)); W = rng.normal(scale=0.05, size=(96, 96))
results = {}
for layers in (2, 4, 8):
    samples = []
    for _ in range(12):
        h = X.copy(); start = time.perf_counter()
        for _ in range(layers): h = np.tanh(h @ W)
        samples.append(time.perf_counter() - start)
    results[layers] = 1000 * float(np.median(samples))
assert all(value > 0 for value in results.values())
print({layers: {"median_forward_ms": round(ms, 4), "time_per_layer_ms": round(ms/layers, 4)} for layers, ms in results.items()})


## 문제 4

**문제**: 임베딩 스케일링 $\sqrt{d}$를 제거하고 학습 안정성을 관찰하라.

### 해설

임베딩 성분 분산이 일정하면 $\sqrt d$ 배율은 잔차 스트림의 초기 크기를 키운다. 제거 효과는 LN 위치와 초기화에 의존하므로 손실 유한성, 그래디언트 노름, 여러 시드의 발산률로 판단한다.


In [ ]:
import numpy as np

rng = np.random.default_rng(1804)
width, batch = 128, 256
embedding = rng.normal(scale=1 / np.sqrt(width), size=(batch, width))
target = rng.normal(size=(batch, width))
results = {}
for scale_name, scale in (("none", 1.0), ("sqrt_d", np.sqrt(width))):
    h = embedding * scale
    loss = float(np.mean((h-target)**2))
    gradient_norm = float(np.linalg.norm(2*(h-target)*scale / h.size))
    results[scale_name] = {"stream_std": float(h.std()), "loss": loss, "gradient_norm": gradient_norm}
assert results["sqrt_d"]["stream_std"] > 8 * results["none"]["stream_std"]
print({k: {m: round(v, 5) for m, v in values.items()} for k, values in results.items()})


## 문제 5

**문제**: 다음 장(Ch 19)에서 이 모델을 학습시킬 것. 사전학습 데이터를 어떻게 준비해야 할지 설계하라.

### 해설

재현 가능한 파이프라인은 라이선스 확인, 정규화·중복 제거, 문서 단위 train/validation 분할, 고정 토크나이저, EOS 연결, 길이 $T+1$의 블록화 순서다. 분할 전에 중복을 제거하여 오염을 막고 데이터 버전·해시·토큰 수를 기록한다.


In [ ]:
import numpy as np

documents = ["attention uses context", "context guides prediction", "tokens form sequences", "sequences train models"]
normalized = [" ".join(doc.lower().split()) for doc in documents]
unique = list(dict.fromkeys(normalized))
split = int(0.75 * len(unique)); train_docs, validation_docs = unique[:split], unique[split:]
vocabulary = {word: i + 1 for i, word in enumerate(sorted(set(" ".join(train_docs).split())))}
eos = 0
train_tokens = [vocabulary[word] for doc in train_docs for word in doc.split()] + [eos]
block_length = 4
blocks = [train_tokens[i:i+block_length+1] for i in range(0, len(train_tokens)-block_length, block_length)]
assert set(train_docs).isdisjoint(validation_docs) and all(len(block) == block_length + 1 for block in blocks)
print({"documents": len(documents), "unique": len(unique), "train_validation": [len(train_docs), len(validation_docs)],
       "vocabulary_size": len(vocabulary)+1, "training_blocks": len(blocks)})
